# Objective: getting some statistics on the product BORAX

In [61]:
from pathlib import Path
import pandas as pd

# Set the data directory as a Path object
DATA_DIR = Path("../data/results")

#-----------------------------------------------------
# RUN THIS BLOCK IF YOU ARE CREATING A NEW DATASET
#-----------------------------------------------------
# Original EXCEL with manual annotation:
#df_labels = pd.read_excel(DATA_DIR / "59ab8652_Borax_de_ch_20250502140755_labeled.xlsx")
df_labels = pd.read_csv(DATA_DIR / "59ab8652_Nitrobenzin_de_ch_20250513101337_labeled.csv")

# Option 1: Read from CSV file from latest analysis
df = pd.read_csv(DATA_DIR / "59ab8652_Nitrobenzin_de_ch_20250513101337_labeled.csv")
df['manual_label'] = df_labels['manual_label']

# Option 2: Read from Excel file (uncomment to use)
#df = pd.read_excel(DATA_DIR / "59ab8652_Borax_de_ch_20250502140755_labeled.xlsx")

# Optionally, display the first few rows
df.head(1).T

,0
search_term,Nitrobenzin
search_term_type,initial
url,https://de.wikipedia.org/wiki/Nitromethan
marketplace_name,Google
domain,de.wikipedia.org
product_name,farblose Flüssigkeit mit fruchtigem Geruch
product_price,NaN
product_description,NaN
product_images,['https://upload.wikimedia.org/wikipedia/commo...
probability,0.010306


In [62]:
# Keep only rows where 'manual_label' is not empty or NaN
df = df[df['manual_label'].notna() & (df['manual_label'] != '')]

# Print the number of rows in the filtered DataFrame
print(f'This is the length of manually labeled products: {len(df)}')

empty_product_name_count = df['product_name'].isna().sum() + (df['product_name'] == '').sum()
print(f'This is the number of empty product names: {empty_product_name_count}')

empty_product_description_count = df['product_description'].isna().sum() + (df['product_description'] == '').sum()
print(f'This is the number of empty product description: {empty_product_description_count}')

empty_product_name_and_description_count = (
    (df['product_name'].isna() | (df['product_name'] == '')) &
    (df['product_description'].isna() | (df['product_description'] == ''))
).sum()
print(f'This is the number of products with both empty product name and product description: {empty_product_name_and_description_count}')

empty_product_name_or_description_count = (
    df['product_name'].isna() | (df['product_name'] == '') |
    df['product_description'].isna() | (df['product_description'] == '')
).sum()
print(f'This is the number of products with either empty product name or product description: {empty_product_name_or_description_count}')

This is the length of manually labeled products: 48
This is the number of empty product names: 5
This is the number of empty product description: 17
This is the number of products with both empty product name and product description: 4
This is the number of products with either empty product name or product description: 18


# Filter out products with empty title/description

### Now what I am going to do is that I want to get ONLY the products where neither of these products is empty - BECAUSE for the time being, this is automatically recognized as -1 and skipped in the pipeline

In [63]:
df = df[
    (~df['product_name'].isna() & (df['product_name'] != '')) &
    (~df['product_description'].isna() & (df['product_description'] != ''))
]
print(f'This is the number of products where neither product name nor product description is empty: {len(df)}')

df['final_relevance'] = df.apply(
    lambda row: 0 if row['seriousness'] == 0.0 else row['relevance'],
    axis=1
)


This is the number of products where neither product name nor product description is empty: 30


In [64]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report

# relevance or seriousness
which_category = 'final_relevance'

# Calculate basic metrics
precision = precision_score(df['manual_label'], df[which_category])
recall = recall_score(df['manual_label'], df[which_category])
f1 = f1_score(df['manual_label'], df[which_category])
accuracy = accuracy_score(df['manual_label'], df[which_category])

# Print individual metrics
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Accuracy: {accuracy:.3f}")

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(df['manual_label'], df[which_category]))
tn, fp, fn, tp = confusion_matrix(df['manual_label'], df[which_category]).ravel()
print("Confusion Matrix Metrics:")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

# Print detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(df['manual_label'], df[which_category]))

Precision: 1.000
Recall: 0.300
F1 Score: 0.462
Accuracy: 0.767

Confusion Matrix:
[[20  0]
 [ 7  3]]
Confusion Matrix Metrics:
True Negatives: 20
False Positives: 0
False Negatives: 7
True Positives: 3

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.74      1.00      0.85        20
           1       1.00      0.30      0.46        10

    accuracy                           0.77        30
   macro avg       0.87      0.65      0.66        30
weighted avg       0.83      0.77      0.72        30



### This is a new attempt

### Remember to change the filename at the end!

In [66]:
blacklist = ['exlibris.ch', 
             'orellfuessli.ch', 
             'lebensmittelkontrolle.lu.ch', 
             'youtube.com',
             'wikipedia.org',
             'webmd.com',
             'migros.ch',
             'help.migros.ch',
             'sciencedirect.com',
             'foodauthority.nsw.gov.au',
             'medicalnewstoday.com',
             'nbcnews.com',
             'orellfuessli.ch',
             'pubchem.ncbi.nlm.nih.gov',
             'exlibris.ch'
            ] 

# Print the number of rows before filtering
print(f"Number of rows before filtering - blacklist websites: {len(df)}")

# Filter out URLs containing blacklisted domains
df_wo_blacklist = df[~df['url'].str.contains('|'.join(blacklist), case=False, na=False)]

# Print the number of rows after filtering
print(f"Number of rows after filtering - blacklist websites: {len(df_wo_blacklist)}")

# Create overwritten_relevance column
df_wo_blacklist['overwritten_relevance'] = df_wo_blacklist['final_relevance'].copy()

# Overwrite relevance to 1.0 where product_price has any value
df_wo_blacklist.loc[df_wo_blacklist['product_price'].notna(), 'overwritten_relevance'] = 1.0

# Display the distribution of values
print("\nDistribution of overwritten_relevance:")
print(df_wo_blacklist['overwritten_relevance'].value_counts().sort_index())

# Display example rows to verify the logic
print("\nExample rows showing the logic:")
print(df_wo_blacklist[['product_price', 'final_relevance', 'overwritten_relevance']].head(3))

# Calculate metrics for overwritten_relevance
y_true = df_wo_blacklist['manual_label']
y_pred = df_wo_blacklist['overwritten_relevance']

# Calculate confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("\nConfusion Matrix Metrics for overwritten_relevance:")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

# Print classification report
print("\nClassification Report for overwritten_relevance:")
print(classification_report(y_true, y_pred))

# Calculate additional metrics
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\nAdditional Metrics for overwritten_relevance:")
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1_score:.3f}")

# Save the dataframe with the new filename
output_filename = str(DATA_DIR / "59ab8652_Nitrobenzin_de_ch_20250513101337_labeled_manual_overwrite.csv")
df_wo_blacklist.to_csv(output_filename, index=False)
print(f"Dataframe saved to: {output_filename}")

Number of rows before filtering - blacklist websites: 30
Number of rows after filtering - blacklist websites: 30

Distribution of overwritten_relevance:
overwritten_relevance
0    22
1     8
Name: count, dtype: int64

Example rows showing the logic:
   product_price  final_relevance  overwritten_relevance
3            NaN                0                      0
4           46.5                0                      1
5            NaN                0                      0

Confusion Matrix Metrics for overwritten_relevance:
True Negatives: 16
False Positives: 4
False Negatives: 6
True Positives: 4

Classification Report for overwritten_relevance:
              precision    recall  f1-score   support

           0       0.73      0.80      0.76        20
           1       0.50      0.40      0.44        10

    accuracy                           0.67        30
   macro avg       0.61      0.60      0.60        30
weighted avg       0.65      0.67      0.66        30


Additional Metri

In [ ]:
###